# Embeddings, Similarity and Semantic Search

Notebook for **Chapter 2** of the *genai* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/genai/ch02.html

Tip: change a number, predict the output, then run the cell again.

## 2.1 From Words to Vectors (Word2Vec, GloVe, FastText)

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
docs = ["the server is slow under heavy load","high latency during peak traffic",
 "response time grows when many users connect","the cat sat on the warm mat",
 "a kitten slept on the sofa","dogs and cats are popular pets",
 "database queries need an index to be fast","slow sql query fixed by adding index",
 "users complain the website is slow","my pet cat likes to sleep"]
tf = TfidfVectorizer(stop_words="english"); X = tf.fit_transform(docs)
svd = TruncatedSVD(n_components=3, random_state=0); E = svd.fit_transform(X)
E = E/np.linalg.norm(E,axis=1,keepdims=True)      # make each vector length 1
print("shape",E.shape)
def search(q,k=3):
    v = svd.transform(tf.transform([q])); v/=np.linalg.norm(v)
    s = E@v[0]
    for i in np.argsort(-s)[:k]: print(f"  {s[i]:.3f}  {docs[i]}")
for q in ["slow website performance","kitten","pet animals"]:
    print(q); search(q)

shape (10, 3)
slow website performance
  0.972  users complain the website is slow
  0.954  the server is slow under heavy load
  0.712  response time grows when many users connect
kitten
  1.000  a kitten slept on the sofa
  0.417  slow sql query fixed by adding index
  0.373  database queries need an index to be fast
pet animals
  1.000  the cat sat on the warm mat
  1.000  my pet cat likes to sleep
  0.088  dogs and cats are popular pets


## 2.2 Contextual Embeddings, BERT and Sentence Transformers

In [ ]:
# not run here: needs `pip install sentence-transformers` and a model download
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("some-embedding-model-name")   # pick one from the docs
vecs = model.encode(["my site is sluggish", "slow response time"],
                    normalize_embeddings=True)
print(vecs.shape)              # (2, dimension)
print(float(vecs[0] @ vecs[1]))  # cosine, because vectors are normalised

## 2.3 Similarity Metrics and Semantic Search

In [ ]:
import numpy as np
a=np.array([1,2,3.]);b=np.array([2,4,6.]);c=np.array([3,-1,0.])
cos=lambda u,v:u@v/np.linalg.norm(u)/np.linalg.norm(v)
print(cos(a,b),cos(a,c), a@b, np.linalg.norm(a-b))

1.0 0.08451542547285165 28.0 3.7416573867739413


## 2.4 Modern Retrieval: Compression, Late Interaction and Rerankers

In [ ]:
n, d = 10_000_000, 1024
for name, bytes_per in [("fp32", 4), ("fp16", 2), ("int8", 1), ("binary", 1 / 8)]:
    print(f"{name:7s} {n * d * bytes_per / 1e9:7.2f} GB")

fp32      40.96 GB
fp16      20.48 GB
int8      10.24 GB
binary     1.28 GB


In [ ]:
import numpy as np
rng = np.random.default_rng(7)
N, D, Q = 5000, 256, 200
scale = 1.0 / np.sqrt(1 + np.arange(D) / 4.0)           # front dims carry more signal
centres = rng.normal(size=(100, D)) * scale             # 100 topics
docs = centres[rng.integers(0, 100, N)] + 0.35 * rng.normal(size=(N, D)) * scale
queries = docs[rng.choice(N, Q, replace=False)] + 0.15 * rng.normal(size=(Q, D)) * scale
def unit(x): return x / np.linalg.norm(x, axis=1, keepdims=True)
docs, queries = unit(docs), unit(queries)
def top10(scores): return np.argsort(-scores, axis=1)[:, :10]
truth = top10(queries @ docs.T)
def recall(found): return np.mean([len(set(a) & set(b)) / 10 for a, b in zip(truth, found)])

for k in (256, 128, 64, 32):
    d2, q2 = unit(docs[:, :k]), unit(queries[:, :k])
    print(f"first {k:3d} dims  bytes/vec fp32={k*4:5d}  recall@10 vs full = {recall(top10(q2 @ d2.T)):.3f}")
lo, hi = docs.min(), docs.max()
qi = lambda x: np.round((x - lo) / (hi - lo) * 255 - 128).astype(np.int8)
sc = qi(queries).astype(np.int32) @ qi(docs).astype(np.int32).T
print(f"int8 (all 256 dims) bytes/vec={D:5d}  recall@10 vs full = {recall(top10(sc)):.3f}")
bd, bq = np.sign(docs), np.sign(queries)
print(f"binary sign bits    bytes/vec={D//8:5d}  recall@10 vs full = {recall(top10(bq @ bd.T)):.3f}")

first 256 dims  bytes/vec fp32= 1024  recall@10 vs full = 1.000
first 128 dims  bytes/vec fp32=  512  recall@10 vs full = 0.914
first  64 dims  bytes/vec fp32=  256  recall@10 vs full = 0.856
first  32 dims  bytes/vec fp32=  128  recall@10 vs full = 0.796
int8 (all 256 dims) bytes/vec=  256  recall@10 vs full = 0.842
binary sign bits    bytes/vec=   32  recall@10 vs full = 0.347


In [ ]:
import numpy as np
rng = np.random.default_rng(7)
N, D, Q = 5000, 256, 200
scale = 1.0 / np.sqrt(1 + np.arange(D) / 4.0)
centres = rng.normal(size=(100, D)) * scale
docs = centres[rng.integers(0, 100, N)] + 0.35 * rng.normal(size=(N, D)) * scale
queries = docs[rng.choice(N, Q, replace=False)] + 0.15 * rng.normal(size=(Q, D)) * scale
unit = lambda x: x / np.linalg.norm(x, axis=1, keepdims=True)
docs, queries = unit(docs), unit(queries)
truth = np.argsort(-(queries @ docs.T), axis=1)[:, :10]
bits_scores = np.sign(queries) @ np.sign(docs).T
for over in (1, 3, 10):
    hits = 0
    for i in range(Q):
        cand = np.argsort(-bits_scores[i])[: 10 * over]          # cheap binary filter
        best = cand[np.argsort(-(docs[cand] @ queries[i]))[:10]]  # fp32 rescore
        hits += len(set(best) & set(truth[i])) / 10
    print(f"fetch {10*over:3d} candidates, rescore, recall@10 = {hits / Q:.3f}")

fetch  10 candidates, rescore, recall@10 = 0.347
fetch  30 candidates, rescore, recall@10 = 0.743
fetch 100 candidates, rescore, recall@10 = 1.000


In [ ]:
import numpy as np
Q = np.array([[1, 0], [0, 1]], float)
A = np.array([[0.9, 0.1], [0.2, 0.8]])
B = np.array([[0.6, 0.6], [0.5, 0.5]])
def maxsim(q, d): return (q @ d.T).max(axis=1).sum()
print("A", round(maxsim(Q, A), 2), "B", round(maxsim(Q, B), 2))
print("single-vector (mean) A:", (Q @ A.mean(0)).sum(), " B:", (Q @ B.mean(0)).sum())

A 1.7 B 1.2
single-vector (mean) A: 1.0  B: 1.1


In [ ]:
print("all:", 1_000_000 * 12 / 1000 / 3600, "hours")
for k in (50, 100, 200):
    print(k, "candidates:", 10 + k * 12, "ms (unbatched)")

all: 3.3333333333333335 hours
50 candidates: 610 ms (unbatched)
100 candidates: 1210 ms (unbatched)
200 candidates: 2410 ms (unbatched)


## Chapter recap

## Practice set

Problem 2.P.1 (easy) Cosine of `[1,0]` and `[0,1]`? **Answer:** 0 (perpendicular).

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/genai/ch02.html